# Property graph validation with HydraPop

[HydraPop](https://github.com/CategoricalData/HydraPop) connects
[Apache TinkerPop](https://tinkerpop.apache.org/) with the translingual programming framework
[Hydra](https://github.com/CategoricalData/hydra). It demonstrates how the same Hydra-generated
validation logic can run in both Java and Python against the same data.

This notebook walks through property graph validation using TinkerPop's built-in
[Modern graph](https://tinkerpop.apache.org/javadocs/current/full/org/apache/tinkerpop/gremlin/tinkergraph/structure/TinkerFactory.html)
in two parts:

- **Part 1: JSON interchange** -- load schemas and graphs from generated JSON files. No server needed.
- **Part 2: Gremlin Server** -- validate a live graph, break it, and see the errors.

## Setup

Run the following from the HydraPop project root to install dependencies and register the
Jupyter kernel:

```bash
pixi install
pixi run register-kernel
```

Then launch Jupyter:

```bash
pixi run jupyter lab
```

When opening this notebook, select the **HydraPop** kernel
(Kernel > Change Kernel > HydraPop).

In [1]:
try:
    import hydrapop
    import hydra.pg.model
except ImportError:
    raise SystemExit(
        "\n\033[1mHydraPop dependencies not found.\033[0m\n\n"
        "This notebook requires the HydraPop pixi environment.\n"
        "From the project root, run:\n\n"
        "    pixi install\n"
        "    pixi run register-kernel\n"
        "    pixi run jupyter lab\n\n"
        "Then select the 'HydraPop' kernel (Kernel > Change Kernel > HydraPop).\n"
        "See https://pixi.sh for pixi installation instructions."
    )

## Part 1: JSON interchange

The Java side of HydraPop is the source of truth for example data. It defines graph schemas and
graphs using Hydra's PG DSL, encodes them as Hydra Terms, and serializes them to JSON.
On the Python side, we decode that JSON and validate using the same Hydra validation logic.

This part requires only the generated JSON files under `src/gen-main/json/` -- no running server.

### Loading the schema

The Modern graph schema defines two vertex types (`person` and `software`) and two edge types
(`knows` and `created`), each with typed properties and required/optional flags.

In [2]:
from hydrapop.decode import load_schema, load_graph
import hydra.pg.validation as pg_validation
from hydra.dsl.python import Just, Nothing
from hydrapop.validate import check_literal, show_literal

schema = load_schema()

for label, vtype in schema.vertices.items():
    props = ", ".join(f"{p.key.value} ({'required' if p.required else 'optional'})" for p in vtype.properties)
    print(f"Vertex type '{label.value}': {props}")
for label, etype in schema.edges.items():
    props = ", ".join(f"{p.key.value} ({'required' if p.required else 'optional'})" for p in etype.properties)
    print(f"Edge type '{label.value}' ({etype.out.value} -> {etype.in_.value}): {props}")

Vertex type 'person': name (required), age (optional)
Vertex type 'software': name (required), lang (required)
Edge type 'created' (person -> software): weight (required)
Edge type 'knows' (person -> person): weight (required)


### Validating the Modern graph

The unmodified Modern graph should pass validation.

In [3]:
graph = load_graph("modern_graph")

result = pg_validation.validate_graph(check_literal, show_literal, schema, graph)
match result:
    case Nothing():
        print("VALID")
    case Just(msg):
        print(f"INVALID - {msg}")

VALID


### Validation error catalog

The Java side generates several "broken" variants of the Modern graph as JSON files,
each introducing a specific validation error. Let's validate them all.

In [4]:
def validate_json(graph_name):
    """Load a graph variant by name and validate it against the schema."""
    graph = load_graph(graph_name)
    result = pg_validation.validate_graph(check_literal, show_literal, schema, graph)
    match result:
        case Nothing():
            print(f"{graph_name}: VALID")
        case Just(msg):
            print(f"{graph_name}: INVALID - {msg}")

test_cases = [
    "modern_graph",
    "missing_required_property",
    "wrong_id_type",
    "unexpected_vertex_label",
    "unexpected_edge_label",
    "property_value_type_mismatch",
    "unexpected_property_key",
    "wrong_in_vertex_label",
    "wrong_out_vertex_label",
    "missing_required_edge_property",
    "unknown_edge_endpoint",
]

for name in test_cases:
    validate_json(name)

modern_graph: VALID
missing_required_property: INVALID - Invalid vertex with id integer:int32:1: Invalid property: Missing value for : name
wrong_id_type: INVALID - Invalid vertex with id string:"notAnInt": Invalid id: expected integer:int32, got string
unexpected_vertex_label: INVALID - Invalid vertex with id integer:int32:100: Unexpected label: robot
unexpected_edge_label: INVALID - Invalid edge with id integer:int32:100: Unexpected label: manages
property_value_type_mismatch: INVALID - Invalid vertex with id integer:int32:1: Invalid property: Invalid value: expected string, got integer:int32
unexpected_property_key: INVALID - Invalid vertex with id integer:int32:1: Invalid property: Unexpected key: favoriteColor
wrong_in_vertex_label: INVALID - Invalid edge with id integer:int32:100: Wrong in-vertex label: expected person, found software
wrong_out_vertex_label: INVALID - Invalid edge with id integer:int32:100: Wrong out-vertex label: expected person, found software
missing_required_

### Inspecting a broken graph

Let's look at the `missing_required_property` variant. Vertex 1 (Marko) should have a
required `name` property, but it was removed.

In [5]:
graph = load_graph("missing_required_property")

for vid, vertex in graph.vertices.items():
    props = {k.value: show_literal(v) for k, v in vertex.properties.items()}
    print(f"  {vertex.label.value} (id={show_literal(vid)}): {props}")

  person (id=integer:int32:1): {'age': 'integer:int32:29'}
  person (id=integer:int32:2): {'age': 'integer:int32:27', 'name': 'string:"vadas"'}
  software (id=integer:int32:3): {'lang': 'string:"java"', 'name': 'string:"lop"'}
  person (id=integer:int32:4): {'age': 'integer:int32:32', 'name': 'string:"josh"'}
  software (id=integer:int32:5): {'lang': 'string:"java"', 'name': 'string:"ripple"'}
  person (id=integer:int32:6): {'age': 'integer:int32:35', 'name': 'string:"peter"'}


## Part 2: Gremlin Server

This part validates a live TinkerPop graph via [gremlinpython](https://pypi.org/project/gremlinpython/).
It requires a running Gremlin Server (3.8.0+) with the Modern graph loaded.

**Gremlin Server setup:**
```bash
# Download Gremlin Server from https://tinkerpop.apache.org/downloads.html
bin/gremlin-server.sh conf/gremlin-server-modern.yaml
```

This starts a server on `ws://localhost:8182/gremlin` with the Modern graph pre-loaded.

### Jupyter compatibility patches

Jupyter runs its own asyncio event loop, but `gremlinpython` uses a synchronous wrapper
(`run_until_complete`) that conflicts with it. We apply three patches:

1. **`nest_asyncio`** -- allows nested `run_until_complete` calls inside Jupyter's loop.
2. **`aiohttp.helpers.TimerContext`** -- aiohttp's timeout context manager requires an active
   asyncio `Task`, which doesn't exist under `nest_asyncio`'s nested execution.
3. **`asyncio.timeouts.Timeout`** -- Python 3.14's built-in `asyncio.timeout` has the same
   task requirement, used by aiohttp's connector.

For both timeout patches, we skip timeout tracking when there is no current task.
This is safe for interactive use; connection errors still propagate normally.

These patches are not needed in a plain Python REPL (e.g. `pixi run console`).

In [6]:
import asyncio
import asyncio.timeouts
import nest_asyncio
nest_asyncio.apply()

# Patch aiohttp's TimerContext to tolerate missing tasks
import aiohttp.helpers
_original_timer_enter = aiohttp.helpers.TimerContext.__enter__

def _patched_timer_enter(self):
    if asyncio.current_task() is None:
        return self
    return _original_timer_enter(self)

aiohttp.helpers.TimerContext.__enter__ = _patched_timer_enter

# Patch asyncio's Timeout to tolerate missing tasks
_original_timeout_aenter = asyncio.timeouts.Timeout.__aenter__

async def _patched_timeout_aenter(self):
    if asyncio.current_task() is None:
        self._state = asyncio.timeouts._State.ENTERED
        return self
    return await _original_timeout_aenter(self)

asyncio.timeouts.Timeout.__aenter__ = _patched_timeout_aenter

### Connecting to Gremlin Server

In [14]:
from gremlin_python.process.anonymous_traversal import traversal
from gremlin_python.driver.driver_remote_connection import DriverRemoteConnection
from gremlin_python.driver.client import Client
from gremlin_python.process.traversal import T
from hydrapop.validate import validate

conn = DriverRemoteConnection('ws://localhost:8182/gremlin', 'g')
g = traversal().with_remote(conn)
client = Client('ws://localhost:8182/gremlin', 'g')

def reset():
    """Reload the Modern graph on the server."""
    client.submit(
        "graph.traversal().V().drop().iterate();"
        "TinkerFactory.generateModern(graph)").all().result()

print("Connected. Vertices:", g.V().count().next())

Connected. Vertices: 6


### Defining the schema with the Python DSL

Here we build the same Modern graph schema programmatically using HydraPop's Python DSL,
rather than loading it from JSON.

In [15]:
from hydrapop.dsl.pg import vertex_type, edge_type, graph_schema, int32, string, float64

person_type = vertex_type("person", int32()).property("name", string(), True).property("age", int32(), False).build()
software_type = vertex_type("software", int32()).property("name", string(), True).property("lang", string(), True).build()
knows_type = edge_type("knows", int32(), "person", "person").property("weight", float64(), True).build()
created_type = edge_type("created", int32(), "person", "software").property("weight", float64(), True).build()
schema = graph_schema([person_type, software_type], [knows_type, created_type])

### Validating the live graph

The `validate(schema, g)` convenience function reads the live graph via the traversal source,
converts it to a Hydra graph, and validates it. It returns a `Result` whose `repr()` is
either `VALID` or `INVALID - ...`.

In [16]:
validate(schema, g)

INVALID - Invalid edge with id integer:int32:27: Wrong out-vertex label: expected person, found software

### Breaking the graph: missing required property

Drop the `name` property from vertex 1 (Marko). Since `name` is required for `person` vertices,
validation should fail.

In [5]:
reset()
g.V(1).properties('name').drop().iterate()
validate(schema, g)

INVALID - Invalid vertex with id integer:int32:1: Invalid property: Missing value for : name

### Breaking the graph: unexpected vertex label

Add a vertex with a label (`robot`) that isn't defined in the schema.

In [6]:
reset()
g.addV('robot').property(T.id, 99).property('name', 'Bender').iterate()
validate(schema, g)

INVALID - Invalid vertex with id integer:int32:99: Unexpected label: robot

### Breaking the graph: wrong property type

Set the `name` property (expected: string) to an integer.

In [7]:
reset()
g.V(1).property('name', 999).iterate()
validate(schema, g)

INVALID - Invalid vertex with id integer:int32:1: Invalid property: Invalid value: expected string, got integer:int32

### Breaking the graph: wrong edge endpoint

Add a `created` edge from Josh to Marko. The schema says `created` goes from `person` to
`software`, but Marko is a `person`.

In [8]:
reset()
josh = g.V(4).next()
marko = g.V(1).next()
g.V(josh).addE('created').to(marko).property(T.id, 99).property('weight', 0.5).iterate()
validate(schema, g)

INVALID - Invalid edge with id integer:int32:99: Wrong in-vertex label: expected software, found person

### Cleanup

In [9]:
conn.close()
client.close()

## Summary

This notebook demonstrated two approaches to property graph validation with HydraPop:

1. **JSON interchange** -- schemas and graphs serialized from Java, decoded and validated in Python.
   No server required. Good for testing and CI.
2. **Gremlin Server** -- live graph validation using gremlinpython, with the schema defined
   using HydraPop's Python DSL. Useful for interactive exploration and debugging.

Both approaches use the same underlying Hydra validation logic, which is generated by Hydra
and guaranteed to be consistent across languages.

For the complete set of validation test cases, see
[test_validation.py](../src/test/python/hydrapop/test_validation.py).

Links:
- [Hydra](https://github.com/CategoricalData/hydra)
- [Apache TinkerPop](https://tinkerpop.apache.org/)
- [HydraPop README](../README.md)